In [180]:
import polars as pl
import re

# citation-claims

In [181]:
claims = pl.read_parquet("output/curator_claims.parquet")
claims.head()

gene_id,sentence_markers,sentence_plain,cited_sentence_marked,claim_plain,anchors,publication_ids,citation_captions,citation_years
str,str,str,str,str,list[struct[2]],list[i64],list[str],list[i64]
"""DDB_G0287681""","""grlR encodes a member of the e…","""grlR encodes a member of the e…","""grlR encodes a member of the e…","""grlR encodes a member of the e…","[{126,[827]}]",[827],"[""Prabu and Eichinger 2006)""]",[2006]
"""DDB_G0290825""","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","[{144,[956]}, {253,[2919]}]","[956, 2919]","[""(Austin et al. 2006)"", ""(Thompson and Kay, 2000)""]","[2006, 2000]"
"""DDB_G0285319""","""There are three histone H3prot…","""There are three histone H3prot…","""There are three histone H3prot…","""There are three histone H3prot…","[{39,[4032]}]",[4032],"[""Bukenberger et al. 1992""]",[1992]
"""DDB_G0292810""","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","[{112,[9912]}]",[9912],"[""Dimond and Loomis 1976)""]",[1976]
"""DDB_G0292206""","""Expression is dependent on a s…","""Expression is dependent on a s…","""Expression is dependent on a s…","""Expression is dependent on a s…","[{126,[6149]}]",[6149],"[""Schatzle et al. 1991)""]",[1991]


In [182]:
# look at a row closely
row = claims.row(1, named=True)
print(row)

{'gene_id': 'DDB_G0290825', 'sentence_markers': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) [[PUB:956]] and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH [[PUB:2919]].', 'sentence_plain': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) (Austin et al. 2006) and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH (Thompson and Kay, 2000).', 'cited_sentence_marked': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) [CITE:956] and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH [CITE:2919].', 'claim_plain': 'The polyketide synthase S

In [183]:
# get preferred column set (same as before)
df = claims.select([
    "claim_plain",
    "anchors",
    "publication_ids",
    "citation_captions",
    "gene_id",
])

In [184]:
# clean up parentheses in citation_captions
df = df.with_columns(
    pl.col("citation_captions")
    .list.eval(
        pl.element()
        .str.replace_all(r"[()]", "")   # remove parentheses
        .str.replace_all(r"\s+", " ")   # collapse whitespace
        .str.strip_chars()              # <-- instead of .str.strip()
    )
    .alias("citation_captions")
)


In [185]:
# clean up very short claims, it shall have at least 6
min_words = 6

df2 = df.with_columns(
    # Count "words" by matching non-space sequences
    pl.col("claim_plain")
      .str.count_matches(r"\S+")
      .alias("n_words")
)

# 1) summary
summary = df2.select([
    pl.len().alias("rows_total"),
    (pl.col("n_words") < min_words).sum().alias("rows_to_drop"),
    (pl.col("n_words") >= min_words).sum().alias("rows_to_keep"),
])
print(summary)
# 2) inspect what you'd drop (optional)
to_drop = (
    df2.filter(pl.col("n_words") < min_words)
       .select(["n_words", "claim_plain", "gene_id"])
       .sort(["n_words", "claim_plain"])
)
with pl.Config(fmt_str_lengths=300):
    display(to_drop.head(50))

# 3) actually filter
df_filtered = df2.filter(pl.col("n_words") >= min_words).drop("n_words")

shape: (1, 3)
┌────────────┬──────────────┬──────────────┐
│ rows_total ┆ rows_to_drop ┆ rows_to_keep │
│ ---        ┆ ---          ┆ ---          │
│ u32        ┆ u32          ┆ u32          │
╞════════════╪══════════════╪══════════════╡
│ 2677       ┆ 24           ┆ 2653         │
└────────────┴──────────────┴──────────────┘


n_words,claim_plain,gene_id
u32,str,str
1,""".""","""DDB_G0284845"""
1,"""pneumoniae.""","""DDB_G0267444"""
1,"""pneumoniae.""","""DDB_G0267630"""
1,"""pneumoniae.""","""DDB_G0279183"""
2,"""(gskA) (..""","""DDB_G0281385"""
…,…,…
5,"""STATa is downregulated by PTP1.""","""DDB_G0281381"""
5,"""Spores have lower cellulose levels.""","""DDB_G0281387"""
5,"""at the tipped aggregate stage.""","""DDB_G0286185"""


In [186]:
# let us examine the duplicated claims closely
# 1) merge duplicates by concatenating gene_id (unique + sorted)
merged = (
    df_filtered
    .group_by([
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions"
    ])
    .agg(
        pl.col("gene_id")
          .unique()
          .sort()
          .str.join(",")
          .alias("gene_id")
    )
    .select([
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions",
        "gene_id",
    ])
)

print("after merge:")
print("rows =", merged.height)
print("unique claim_plain =", merged.select(pl.col("claim_plain").n_unique()).item())

# 2) check if claim_plain is STILL duplicated (i.e., same claim text but different citation/anchors/etc.)
still_dups = (
    merged
    .with_columns(pl.len().over("claim_plain").alias("n"))
    .filter(pl.col("n") > 1)
    .sort(["claim_plain"])
)

print("still duplicated claim_plain rows =", still_dups.height)

# show them grouped together if any remain
still_dups

after merge:
rows = 2107
unique claim_plain = 2103
still duplicated claim_plain rows = 8


claim_plain,anchors,publication_ids,citation_captions,gene_id,n
str,list[struct[2]],list[i64],list[str],str,u32
"""In Dictyostelium discoideum an…","[{183,[5046]}]",[5046],"[""Hauser et al. 1995""]","""DDB_G0267402,DDB_G0270838,DDB_…",2
"""In Dictyostelium discoideum an…","[{183,[5046]}]",[5046],"[""Hauser and colleagues 1995""]","""DDB_G0285319""",2
"""The 2C gene as well as 5 relat…","[{149,[1381]}]",[1381],"[""Schilde et al. 2004""]","""DDB_G0280847""",2
"""The 2C gene as well as 5 relat…","[{149,[10129]}]",[10129],"[""Schilde et al. 2004""]","""DDB_G0280871,DDB_G0280953,DDB_…",2
"""These mutant cells are also im…","[{148,[1548]}]",[1548],"[""Zhang et al. 2003""]","""DDB_G0279413""",2
"""These mutant cells are also im…","[{148,[2842]}]",[2842],"[""Wessels et al. 2000""]","""DDB_G0284331""",2
"""abcF1 and abcF4 are the most c…","[{115,[8189]}]",[8189],"[""Anjard and Loomis 2002""]","""DDB_G0267436,DDB_G0275637,DDB_…",2
"""abcF1 and abcF4 are the most c…","[{115,[2279]}]",[2279],"[""Anjard and Loomis 2002""]","""DDB_G0285997""",2


In [187]:
# with pl.Config(fmt_str_lengths=60):
#     display(still_dups)       


There are some duplicated claims.

Except the first one, the others are claims with inconsistant citations, let us remove those except the first one.

In [188]:
# 1) add a stable row id to merged
merged_i = merged.with_row_count("row_nr")   # if this errors on your Polars, use: .with_row_index("row_nr")

# 2) recompute still_dups from merged_i (so it contains row_nr)
still_dups_i = (
    merged_i
    .with_columns(pl.len().over("claim_plain").alias("n"))
    .filter(pl.col("n") > 1)
    .sort(["claim_plain", "row_nr"])
)

# 3) drop everything in still_dups except the first row of still_dups
drop_row_nrs = still_dups_i.slice(1).select("row_nr")   # everything after the first row

merged_clean = (
    merged_i
    .join(drop_row_nrs, on="row_nr", how="anti")  # remove those rows
    .drop(["row_nr", "n"], strict=False)
)


/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_50959/3282369987.py:2: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  merged_i = merged.with_row_count("row_nr")   # if this errors on your Polars, use: .with_row_index("row_nr")


In [189]:
merged_clean

claim_plain,anchors,publication_ids,citation_captions,gene_id
str,list[struct[2]],list[i64],list[str],str
"""PKA is also part of the signal…","[{178,[3792]}]",[3792],"[""Souza et al. 1998""]","""DDB_G0283907"""
"""The heavy chain has a stem at …","[{166,[12110]}]",[12110],"[""Kon et al. 2012""]","""DDB_G0276355"""
"""RegA is part of the circuit th…","[{93,[3726]}, {94,[1420]}]","[3726, 1420]","[""Laub et al. 1998"", ""Maeda et al. 2004""]","""DDB_G0284331"""
"""rasD mutants exhibit similar p…","[{189,[17969]}]",[17969],"[""Gruenheit et al. 2018""]","""DDB_G0269252"""
"""In vitro, profilin I raises th…","[{143,[6071]}]",[6071],"[""Haugwitz et al. 1991""]","""DDB_G0287125"""
…,…,…,…,…
"""EGF repeats are found in prote…","[{196,[2172]}]",[2172],"[""Fey et al. 2002""]","""DDB_G0288511"""
"""The sevencalcium up-regulated …","[{100,[1293]}]",[1293],"[""Coukell et al. 2004""]","""DDB_G0267466,DDB_G0272242,DDB_…"
"""Protein-tyrosine kinase genes …","[{119,[6479]}, {120,[4724]}]","[6479, 4724]","[""Tan and Spudich, 1990"", ""Adler et al. 1996""]","""DDB_G0283385"""


In [190]:
# give claim ID, and we want to explode the list columns together to get one row per (claim_id, publication_id)
m = merged_clean.with_columns(
    pl.col("claim_plain").rank(method="dense").cast(pl.Int64).alias("claim_id")
).select([
    "claim_id",
    "claim_plain",
    "anchors",
    "publication_ids",
    "citation_captions",
    "gene_id"
])

In [191]:
# 2) sanity check: list columns must be aligned per row before explode (len(citation_captions) == len(publication_ids))
bad_rows = (
    m
    .with_columns([
        pl.col("publication_ids").list.len().alias("n_pub"),
        pl.col("citation_captions").list.len().alias("n_cap")
    ])
    .filter(
        (pl.col("n_pub") != pl.col("n_cap")) 
    )
    .select([
        "claim_id",
        "gene_id",
        "n_pub", "n_cap", 
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions"
    ])
    .sort(["n_pub", "n_cap"], descending=True)
)

print("Number of misaligned rows:", bad_rows.height)

with pl.Config(fmt_str_lengths=1000):
    display(bad_rows)

Number of misaligned rows: 4


claim_id,gene_id,n_pub,n_cap,claim_plain,anchors,publication_ids,citation_captions
i64,str,u32,u32,str,list[struct[2]],list[i64],list[str]
1534,"""DDB_G0287031""",2,1,"""These results reveal that gpaC functions as a regulator of early development gene expression,.""","[{92,[4035]}, {93,[4034]}]","[4035, 4034]","[""Brandon et al. 1997""]"
1469,"""DDB_G0293084""",2,1,"""The zizB mutant phenotypes and ZizB binding partners suggest a central role for ZizB in actin cytoskeletal organization and cortical stabilization.,.""","[{147,[12098]}, {148,[12830]}]","[12098, 12830]","[""Pakes et al. 2012""]"
1195,"""DDB_G0286183""",2,1,"""The Dictyostelium Agps protein has been crystallized.""","[{52,[712, 659]}]","[712, 659]","[""Razeto et al ., 2007""]"
471,"""DDB_G0268620""",2,1,"""Fluid uptake by macropinocytosis is reduced about 3 fold and phagosomes remain more acidic in pkbA - null cells (, (.""","[{113,[2428]}, {116,[2634]}]","[2428, 2634]","[""Rupper et al. 2001""]"


in the 4 cases here, len(publication_ids)>len(citation_captions), after closer look, it is the same author and same year, we manully make a patch here

In [194]:
m_fixed = (
    m.with_columns(
        pl.when(pl.col("claim_id") == 1534)
          .then(pl.lit(["Brandon et al. 1997a", "Brandon et al. 1997b"]))
        .when(pl.col("claim_id") == 471)
          .then(pl.lit(["Rupper et al. 2001a", "Rupper et al. 2001b"]))
        .when(pl.col("claim_id") == 1469)
          .then(pl.lit(["Pakes et al. 2012a", "Pakes et al. 2012b"]))
        .when(pl.col("claim_id") == 1195)
          .then(pl.lit(["Razeto et al. 2007a", "Razeto et al. 2007b"]))
        .otherwise(pl.col("citation_captions"))
        .alias("citation_captions")
    )
)

In [195]:
m_long = (
    m_fixed
    .explode(["publication_ids", "citation_captions"])
    .rename({"publication_ids": "publication_id"})
)

In [196]:
m_long

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id
i64,str,list[struct[2]],i64,str,str
947,"""PKA is also part of the signal…","[{178,[3792]}]",3792,"""Souza et al. 1998""","""DDB_G0283907"""
1327,"""The heavy chain has a stem at …","[{166,[12110]}]",12110,"""Kon et al. 2012""","""DDB_G0276355"""
1083,"""RegA is part of the circuit th…","[{93,[3726]}, {94,[1420]}]",3726,"""Laub et al. 1998""","""DDB_G0284331"""
1083,"""RegA is part of the circuit th…","[{93,[3726]}, {94,[1420]}]",1420,"""Maeda et al. 2004""","""DDB_G0284331"""
2043,"""rasD mutants exhibit similar p…","[{189,[17969]}]",17969,"""Gruenheit et al. 2018""","""DDB_G0269252"""
…,…,…,…,…,…
1446,"""The sevencalcium up-regulated …","[{100,[1293]}]",1293,"""Coukell et al. 2004""","""DDB_G0267466,DDB_G0272242,DDB_…"
999,"""Protein-tyrosine kinase genes …","[{119,[6479]}, {120,[4724]}]",6479,"""Tan and Spudich, 1990""","""DDB_G0283385"""
999,"""Protein-tyrosine kinase genes …","[{119,[6479]}, {120,[4724]}]",4724,"""Adler et al. 1996""","""DDB_G0283385"""


In [197]:
# now we exlpode anchors
# 1) Build mapping: (claim_id, publication_id) -> anchor_pos
anchor_map = (
    m_fixed
    .select(["claim_id", "anchors"])
    .explode("anchors")  # now each row is one struct {pos, pub_ids}
    .with_columns([
        pl.col("anchors").struct.field("pos").alias("anchor_pos"),
        pl.col("anchors").struct.field("pub_ids").alias("publication_id"),
    ])
    .explode("publication_id")  # one row per pub_id
    .select(["claim_id", "publication_id", "anchor_pos"])
)

# If there can be multiple anchor_pos for the same (claim_id, publication_id), keep them all:
anchor_map = (
    anchor_map
    .group_by(["claim_id", "publication_id"])
    .agg(pl.col("anchor_pos").sort().alias("anchor_pos"))
)

# 2) Join onto m_long
m_long2 = m_long.join(anchor_map, on=["claim_id", "publication_id"], how="left")

# 3) If you want exactly one row per (claim_id, publication_id, anchor_pos), explode anchor_pos:
m_long3 = m_long2.explode("anchor_pos")

# Final columns (example)
m_long3.select([
    "claim_id",
    "claim_plain",
    "anchor_pos",
    "publication_id",
    "citation_captions",
    "gene_id",
])

claim_id,claim_plain,anchor_pos,publication_id,citation_captions,gene_id
i64,str,i64,i64,str,str
947,"""PKA is also part of the signal…",178,3792,"""Souza et al. 1998""","""DDB_G0283907"""
1327,"""The heavy chain has a stem at …",166,12110,"""Kon et al. 2012""","""DDB_G0276355"""
1083,"""RegA is part of the circuit th…",93,3726,"""Laub et al. 1998""","""DDB_G0284331"""
1083,"""RegA is part of the circuit th…",94,1420,"""Maeda et al. 2004""","""DDB_G0284331"""
2043,"""rasD mutants exhibit similar p…",189,17969,"""Gruenheit et al. 2018""","""DDB_G0269252"""
…,…,…,…,…,…
1446,"""The sevencalcium up-regulated …",100,1293,"""Coukell et al. 2004""","""DDB_G0267466,DDB_G0272242,DDB_…"
999,"""Protein-tyrosine kinase genes …",119,6479,"""Tan and Spudich, 1990""","""DDB_G0283385"""
999,"""Protein-tyrosine kinase genes …",120,4724,"""Adler et al. 1996""","""DDB_G0283385"""


In [198]:
# now we get year from citation_captions
YEAR_RE = r"(18|19|20)\d{2}"

m_long3 = m_long3.with_columns(
    year=pl.col("citation_captions")
        .str.extract(YEAR_RE, 0)   # whole match
        .cast(pl.Int32)
)
m_long3

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year
i64,str,list[struct[2]],i64,str,str,i64,i32
947,"""PKA is also part of the signal…","[{178,[3792]}]",3792,"""Souza et al. 1998""","""DDB_G0283907""",178,1998
1327,"""The heavy chain has a stem at …","[{166,[12110]}]",12110,"""Kon et al. 2012""","""DDB_G0276355""",166,2012
1083,"""RegA is part of the circuit th…","[{93,[3726]}, {94,[1420]}]",3726,"""Laub et al. 1998""","""DDB_G0284331""",93,1998
1083,"""RegA is part of the circuit th…","[{93,[3726]}, {94,[1420]}]",1420,"""Maeda et al. 2004""","""DDB_G0284331""",94,2004
2043,"""rasD mutants exhibit similar p…","[{189,[17969]}]",17969,"""Gruenheit et al. 2018""","""DDB_G0269252""",189,2018
…,…,…,…,…,…,…,…
1446,"""The sevencalcium up-regulated …","[{100,[1293]}]",1293,"""Coukell et al. 2004""","""DDB_G0267466,DDB_G0272242,DDB_…",100,2004
999,"""Protein-tyrosine kinase genes …","[{119,[6479]}, {120,[4724]}]",6479,"""Tan and Spudich, 1990""","""DDB_G0283385""",119,1990
999,"""Protein-tyrosine kinase genes …","[{119,[6479]}, {120,[4724]}]",4724,"""Adler et al. 1996""","""DDB_G0283385""",120,1996


In [199]:
year_summary = (
    m_long3
    .with_columns(pl.col("year").fill_null(-1).alias("year2"))
    .group_by("year2")
    .agg(pl.len().alias("n"))
    .sort("year2")
    .with_columns(
        pl.when(pl.col("year2") == -1).then(None).otherwise(pl.col("year2")).alias("year")
    )
    .select(["year", "n"])
)

year_summary


year,n
i32,u32
null,7
1956,1
1965,1
1967,3
1968,1
…,…
2016,77
2017,60
2018,81


In [200]:
claim_cleaned = m_long3.select([
    "claim_id",
    "claim_plain",
    "anchors",        # keep if you still want the original struct list
    "anchor_pos",
    "publication_id",
    "citation_captions",
    "gene_id",
    "year",
])

In [209]:
claim_cleaned.write_parquet("output/cleaned/claim_cleaned_long.parquet")

In [210]:
claim_cleaned.select(
    pl.col("claim_id").n_unique().alias("n_unique_claim_id")
)


n_unique_claim_id
u32
2100


In [211]:
# to_save = claim_cleaned.with_columns(
#     pl.col("anchors").cast(pl.Utf8)   # converts list/struct to a string representation
# ).select([
#     "claim_id","claim_plain","anchors","anchor_pos",
#     "publication_id","citation_captions","gene_id","year"
# ])

# to_save.write_csv("output/cleaned/claim_cleaned_long.tsv", separator="\t")


I will certainly need to still manually check out the datasets later. So far we have 2100 unique claims.

In [208]:
# manual inspection, will pick some golden set maunally later
claim_cleaned_manual = claim_cleaned.filter(pl.col("anchor_pos") != 1)

# match pmid

In [213]:
pub_pmid = pl.read_csv("output/publication_id_pmid.csv")
pub_pmid

publication_id,pmid
i64,i64
12,21243421
13,21239624
15,21235525
17,20950684
20,21150268
…,…
19689,32769116
19708,21551065
19728,32821814


In [217]:
# check if all publcation id can be mapped to pmid

# pick the right df names
a = claim_cleaned
b = pub_pmid

# make sure both publication_id columns are the same type (I recommend Int64)
a_ids = a.select(pl.col("publication_id").cast(pl.Int64)).unique()
b_ids = b.select(pl.col("publication_id").cast(pl.Int64)).unique()

# overlap + only-in sets
overlap = a_ids.join(b_ids, on="publication_id", how="inner")
only_a  = a_ids.join(b_ids, on="publication_id", how="anti")
only_b  = b_ids.join(a_ids, on="publication_id", how="anti")

summary = pl.DataFrame({
    "set": ["claim_cleaned", "pub_pmid", "overlap", "only_claim_cleaned", "only_pub_pmid"],
    "unique_count": [a_ids.height, b_ids.height, overlap.height, only_a.height, only_b.height],
})

summary

set,unique_count
str,i64
"""claim_cleaned""",1400
"""pub_pmid""",4341
"""overlap""",1373
"""only_claim_cleaned""",27
"""only_pub_pmid""",2968


25 ids are not mapped.

In [219]:
claim_cleaned_pmid = (
    claim_cleaned
    .with_columns(pl.col("publication_id").cast(pl.Int64))
    .join(pub_pmid, on="publication_id", how="left")
    .with_columns(
        pl.col("pmid").fill_null("NA")  # or keep as null if you prefer
    )
)

claim_cleaned_pmid

claim_id,claim_plain,anchors,anchor_pos,publication_id,citation_captions,gene_id,year,pmid
i64,str,list[struct[2]],i64,i64,str,str,i32,str
947,"""PKA is also part of the signal…","[{178,[3792]}]",178,3792,"""Souza et al. 1998""","""DDB_G0283907""",1998,"""9584128"""
1327,"""The heavy chain has a stem at …","[{166,[12110]}]",166,12110,"""Kon et al. 2012""","""DDB_G0276355""",2012,"""22398446"""
1083,"""RegA is part of the circuit th…","[{93,[3726]}, {94,[1420]}]",93,3726,"""Laub et al. 1998""","""DDB_G0284331""",1998,"""9843585"""
1083,"""RegA is part of the circuit th…","[{93,[3726]}, {94,[1420]}]",94,1420,"""Maeda et al. 2004""","""DDB_G0284331""",2004,"""15131307"""
2043,"""rasD mutants exhibit similar p…","[{189,[17969]}]",189,17969,"""Gruenheit et al. 2018""","""DDB_G0269252""",2018,"""30473004"""
…,…,…,…,…,…,…,…,…
1446,"""The sevencalcium up-regulated …","[{100,[1293]}]",100,1293,"""Coukell et al. 2004""","""DDB_G0267466,DDB_G0272242,DDB_…",2004,"""14871937"""
999,"""Protein-tyrosine kinase genes …","[{119,[6479]}, {120,[4724]}]",119,6479,"""Tan and Spudich, 1990""","""DDB_G0283385""",1990,"""1972546"""
999,"""Protein-tyrosine kinase genes …","[{119,[6479]}, {120,[4724]}]",120,4724,"""Adler et al. 1996""","""DDB_G0283385""",1996,"""8898113"""


In [220]:
claim_cleaned_pmid.write_parquet("output/cleaned/claim_cleaned_long_pmids.parquet")

claim_cleaned_pmid_nonNA = claim_cleaned_pmid.filter(
    pl.col("pmid").is_not_null() & (pl.col("pmid") != "NA")
)

claim_cleaned_pmid_nonNA.write_parquet("output/cleaned/claim_cleaned_long_pmids_nonNA.parquet")


In [222]:
claim_cleaned_pmid_nonNA.select(
    pl.col("claim_id").n_unique().alias("n_unique_claim_id")
)


n_unique_claim_id
u32
2077


# how many we have abstracts on EPMC

In [226]:
EPMC = pl.read_parquet("output/cleaned/articles_all_cleaned_abstract.parquet")

In [227]:
EPMC

pmid,pmcid,doi,year,title,journal,authors,abstract_clean,file
str,str,str,str,str,str,str,str,str
"""2654141""","""PMC2115546""","""10.1083/jcb.108.5.1751""","""1989""","""Centrin-mediated microtubule s…","""The Journal of cell biology""","""Sanders MA, Salisbury JL.""","""Chlamydomonas cells excise the…","""article_fetching/output/all_cl…"
"""39528565""","""PMC11555045""","""10.1038/s41467-024-54272-4""","""2024""","""Nuclear localization sequence …","""Nature communications""","""Lim YJ, Yoon YJ, Lee H, Choi G…","""Plant pathogens secrete nuclea…","""article_fetching/output/all_cl…"
"""6319129""",null,"""10.1111/j.1432-1033.1984.tb078…","""1984""","""Investigations on stimulation …","""European journal of biochemist…","""Scholübbers HG, van Knippenber…","""The ability of 24 systematical…","""article_fetching/output/all_cl…"
"""28740830""","""PMC5502327""","""10.3389/fonc.2017.00139""","""2017""","""Structure, Activity Regulation…","""Frontiers in oncology""","""Mammucari C, Gherardi G, Rizzu…","""Mitochondrial Ca2+ uptake play…","""article_fetching/output/all_cl…"
"""18814278""",null,"""10.1002/cm.20314""","""2008""","""Correlated waves of actin fila…","""Cell motility and the cytoskel…","""Asano Y, Nagasaki A, Uyeda TQ.""","""Chemotaxis-deficient amiB-null…","""article_fetching/output/all_cl…"
…,…,…,…,…,…,…,…,…
"""20023070""","""PMC2823002""","""10.1128/ec.00220-09""","""2010""","""Distinct subcellular localizat…","""Eukaryotic cell""","""Schilde C, Schönemann B, Sehri…","""We have identified new synapto…","""article_fetching/output/all_cl…"
"""10706824""",null,"""10.1006/scdb.1999.0343""","""1999""","""Control of spatial patterning …","""Seminars in cell & development…","""Mohanty S, Firtel RA.""","""The spatial patterning of pres…","""article_fetching/output/all_cl…"
null,null,"""10.1101/2022.10.21.513250""","""2022""","""Loss of altruism in the social…",null,"""Walker LM, Sherpa RN, Ivaturi …","""Aggregative multicellularity r…","""article_fetching/output/all_cl…"
